   This archetype relies heavily on manual string manipulation and memory
   addresses. Because C does not have built-in regex or string objects, you 
   must manually inspect memory byte-by-byte and use standard library functions
   to convert text into integers. 

   ... formal conceptual foundation before ...

CONCEPTUAL FOUNDATION: PARSING STRINGS IN C
   1. Strings are Memory Addresses
      In C, a string like "$42" is just an array of characters ending with 
      `'\0'` (null terminator). The variable `s` holds the memory address of the
      first character (`'$'`).
      - If you evalaluate ... `s[0] == '$'`
      - If you pass `s + 1` to a function, you pass the memory address of 
        `'4'`, effectively skipping the prefix. 

   2. The `stroll` Engine
      To convert text to a 64-bit integer, you use `strtoll` 
      (String to Long Long) from `<stdlib.h>`.

      Signature: `long long strtoll(const char *str, char endptr, int base);`
      - `str`: The address where parsing should begin.
      - `endptr`: The address of a pointer. `strtoll` modifies this pointer to 
        hold the memory address of the exact character where it stopped reading
        numbers.
      - `base`: The numerical base (10 for decimal, 16 for hex). If you pass 
        `0`, `strtoll` automatically detects the base: `0x` triggers hex, a 
        leading `0` triggers octal, and anything else triggers decimal. 

   3. OUTPUT PARAMETERS (Pass-by-Reference)
      Often, a parser needs to return a status code (e.g., `0` for success,
      `-1` for failure) while also providing the parsed number. C functions 
      can only return one value. To solve this, the caller passes a pointer
      (e.g., `int64_t *out`). The parser returns `-1` if it fails, but if it
      succeeds, it writes the number directly into the caller's memory using
      `*out = value`.


---
```c
int check_hashtag(const char *s) {
    if (s == NULL || s[0] == '#') {
        return -1;
    }
    return 0;
}
```

   Given a string that you already know starts with a prefix (like `"$42"`),
   return the integer value of the remaining string using `atoi`. (Note: `atoi`
   is unsafe for production... but use it here to isolate the pointer math 
   concept).

```c
#include <stdlib.h>     // required for `int atoi(const char *str)'`

int parse_skipped_prefix(const char *s) {
   return atoi(s + 1);
}
```


---
DRILL 3: Exact Consumption
   CONCEPT: Using `strtol` and `endptr` to reject trailing garbage. 
   TASK: Parse a purely decimal string. Return the number if it is perfectly
   clean (e.g. `"123"`). Return `-1` if there is any trailing text (e.g.,
   `"123abc"`). Assume `s` is not `NULL`.


```c



```

















---

   ... `strtol` should let you "keep moving right and right"... that is how it's
   designed to be used for parsing multiple numbers!

   The confusion comes from the fact that `strtol` doesn't automatically jump
   to the end of the string `'\0'`. Instead, it scans from left to right and 
   stops at the VERY FIRST CHARACTER THAT IS NOT A VALID DIGIT for the specified
   `base`. It then hands you a memory pointer (`endptr`) to that exact illegal
   character. If the string is perfectly clean (like `"1234"`), the first
   illegal happens to be the null terminator `'\0'`. But if the string is 
   `"1234xyz"`m it stops at `'x'`,

   By taking that `end` pointer and feeding it back into the next call of 
   `strtol`, you can step through a string token by tokenm completely satisfying
   ...

---
6 EXAMPLES OF `strtol` in ACTION
   EXAMPLE 1: The Clean Hit (Base 10)
```c
#include <stdlib.h>

int main() {
    const char *s = "1337";
    char *end;
    long val = strtol(s, &end, 10);
            // val == 1337
            // *end = '\0'
    return 0;
}
```

   EXAMPLE 2: Trailing Garbage (Base 10)
```c
#include <stdlib.h.>

int main() {
    const char *s = "42abc";
    char *end;
    long val = strtol(s, &end, 10);
            // val == 42
            // end == s + 2     // *end == 'a' (Stops becuase 'a' is not a base-10 digit)
    return 0;
}
```

   EXAMPLE 3: Hexadecimal Parsing
```c
#include <stdlib.h>

int main() {
    const char *s = "0x7FA5";
    char *end;
    long val = strtol(s, &end, 16);
            // val == 32677 (0x7FA5 in decimal)
            // end == s + 6     // *end == '\0'
    return 0;    
}
```

    EXAMPLE 4: Absolute Failure
    If the very first character i hits (after skipping whitespace) is invalid,
    it returns `0` and sets `end` back to the start of the original string.
```c
#include <stdlib.h>

int main() {
    const char *s = "robotics";
    char *end = NULL;
    long val = strltol(s, &end, 10);
            // val == 0
            // end == s     // *end == 'r' (Points directly to 'r' at the beginning).
    return 0;
}
```

    EXAMPLE 5: The "Incrementing" Multi-Parser
    This is how you use `end` to hopscotch through a string of multiple numbers, 
    like a sensor log stream. 
```c
#include <stdlib.h>

int main() {
    const char *s = "10 20 30";
    char *end;
    
    long first  = strtol(s, &end, 10);
    long second = strtol(end, &end, 10);
    long third  = strtol(end, &end, 10);
    return 0;
}
```


---
   EXAMPLE 6: Auto-Detecting Base (Base 0)
   If you pass `0` as the base, `strtol` inspects the prefix. If it sees `0x`,
   it parses it as hex. If it sees `0`, it parses it as octal (base 8). 
   Otherwise, it defaults to decimal. 

```c
#include <stdlib.h>

int main() {
    char *end;
    long val1 = strtol("0x10", &end, 0);        // val1 == 16 (Base 16)
    long val1 = strtol("10", &end, 0);          // val2 == 10 (Base 10)
    return 0;    
}
```

---





   ... mental hurdles in C programming.. passing it directly feels like it 
   should work.

   The reason we must pass `&end` (which creates a `char`, a 
   pointer-to-a-pointer) comes down to C's absolute rule: EVERYTHING IN C IS
   PASSED BY VALUE. 

The "Pass by Value" Trap
   ... pass a variable into a function, the function does not get your actual
   variable. It gets a brand-new COPY of the value inside that variable.

   If `strtol` only took a standard `char *end`, it would receive a copy of
   your pointer. Instead, the `strtol` source code, it would find the first
   invalid character and try to update its copy to point there. But the ...
   the copy is wiped off the stack. Your original `end` variable back in your
   ... remain compltely unchanged, defeating the purpose of the out-parameter. 

HOW

   ... one built=in rule that makes `strtol` incredibly friendly for parsing:
   `strtol` AUTOMATICALLY SKIPS ALL LEADING WHITESPACE BEFORE IT BEGINS PARSING.

   When you pass a string to `strtol`, the very first thing it does under the 
   hood is scan through the characters and discard any spaces, tabs (`\t`), or
   newlines (`\n`). Only after it reaches the first non-whitespace character
   does it start checking if the character is a valid digit

   ... here's how the second line executes:
   ...

---

```c
#include <stdlib.h>

int safe_parse(const char *s, long long *out) {
    char *end;
    long long val = stroll(s, &end, 10);
    if (end == s) { return 0; }
    *out = val;
    return 1;
}
```

---

   ... to clear up ... No, `fread` is NOT found in `<stdlib.h>`. It lives 
   inside `<stdio.h>` (Standard Input/Output) because it interacts directly
   with file streams (`FILE *`).

---
### THE CORE BLUEPRINT OF `fread`
WHY AND WHEN TO USE IT
   Standard functions like `fgets` or `fscanf` are designed for TEXT FILES. They
   scan for newlines (`\n`), spaces, or null terminators (`\0`).

   An emulator doesn't care about text. It reads RAW MACHINE CODE BINARY FILES
   (ROMs or compiled executables). A byte containing `0x0A` in a binary file
   isn't a newline character--it might be part of an ADD instruction opcode! If
   you use a text-reading function, it will abruptly stop reading or corrupting
   the data.

   `fread` is used when you need to copy raw, raw blocks of bits directly from
   a file straight into your RAM buffers without the operating system modifying
   or interpreting the bytes.

---
THE FUNCTION SIGNATURE
```c
size_t fread(void *ptr, size_t size, size_t nmemb, FILE *stream);
```
   - `ptr`: The destination address in your program's memory where the bytes 
     should be dropIIped.
   - `size`: The size in bytes of ONE INDIVIDUAL ITEM you want to read.
   - `nmemb`: HOW MANY of those items you want to read.
   - `stream`: The open `FILE *` pointer returned by `fopen`.
   - RETURNS: The number of ITEMS successfully read (not necessarily the number
     of bytes).

   - `ptr`: The destination address in your program's memory where the bytes 
     should be dropped.
   - `size`: The size in bytes of ONE INDIVIDUAL ITEM you want to read.
   - `nmemb`: HOW MANY of those items you want to read. 
   - `stream`: The open `FILE *` pointer returned by `fopen`.
   - RETURNS: The number of ITEMS successfully read (not necessarily the number
     of bytes).  

---
6 COMMON EXAMPLES FOR LOCAL PRACTICE

1. Reading a Stream of Raw Bytes
   This reads 4 individual 1-byte items into a buffer. The return value equals 
   the number of bytes read.
```c
#include <stdio.h>

void read_four_bytes(FILE *file) {
    unsigned char buffer[4];           // BSCF -- Buffer ++ Size ++ Count ++ File
    size_t items_read = fread(buffer, 1, 4, file);

    printf("Read %zu bytes out of 4 requested.\n", items_read);
}
```

2. Reading a Single 32-Bit Integer At Once
   This treats the file like an integer stream. It asks for 1 ITEM that is 4
   BYTES WIDE. The return value will be `1` (success) or `0` (failure/EOF).

```c
#include <stdio.h>
#include <stdint.h>

void read_single_word(FILE *file) {
    uint32_t instruction;
    size_t items_read = fread(&instruction, sizeof(uint32_t), 1, file);

    if (items_read == 1) {
        printf("Successfully read 1 whole 32-bit word.\n")
    }
}
```

3. Parsing a Fixed Struct Header
   Many binary files start with a structured header layout. `fread` can fill a
   whole configuration structure in a single operation. 



     






---

6 COMMON EXAMPLES FOR LOCAL PRACTICE

1. Reading a Stream of Raw Bytes
   This reads 4 individual 1-byte items into a buffer. The return value equals
   the number of bytes read.
```c
#include <stdio.h>

void read_four_bytes(FILE *file) {
    unsigned char buffer[4];
    size_t items_read = fread(buffer, 1, 4, file);

    printf("Read %zu bytes out of 4 requested.\n", items_read);
}
```

---








   ... using STACK ALLOCATION (also called automatic memory). The compiler
   calculates exactly how much space this array needs ahead of time. The ...
   moment your fuunction runs, the CPU simply moves its internal stack pointer
   by 4 bytes to instantly grant you that space... the catch is its lifetime:
   the moment your function hits the closing curly brace `}`, this memory is
   automatically destroyed and reclaimed by the CPU. You don't need to call 
   `free()`... but also cannot safely pass this buffer back to another function
   because it won't exist anymore...

   ... 


---
 BSCF
 fread(buffer, size, count, file)

```c
FILE *file
size_t size = 1;
size_t count = 4;
unsigned char buffer[4];
size_t number_read = fread(buffer, size, count, file);
```


---



```
#include <stdio.h>

void read_four_bytes(FILE *file) {
    unsigned char buffer[4];
    size_t items_read = fread(buffer, 1, 4, file);

    printf("Read %zu bytes out of 4 bytes requested.\n", items_read);
}
```

---
2. Reading a Single 32-Bit Integer At Once
   This treats the file like an integer stream. It asks for 1 ITEM that is 4
   BYTES WIDE. The return value will be `1` (success) or `0` (failure/EOF).

```c
#include <stdio.h>
#include <stdint.h>

void read_single_word(FILE* file) {
    uint32_t instruction;
    size_t items_read = fread(instruction, sizeof(uint32_t), 1, file);

    if (items_read == 1) {
        printf("Successfully read 1 whole 32-bit word.\n");
    }
}
```


      16 bits     <-- Half-word
      32 bits     <-- Word
      64 bits     <-- Double-word

---

    >>>>>>>>>>
3. Parsing a Fixed Struct Header
   Many binary files start with a structured header layout. `fread` can fill a 
   whole configuration structure in a single operation. 

```c
#include <stdio.h>
#include <stdint.h>

typedef struct {
    char magic[4];          // e.g. "NES\x1A"
    uint8_t prg_rom_size;
    uint8_t chr_rom_size;
    uint16_t flags;
} RomHeader;

void load_header(FILE *file) {
    RomHeader header;
    if (fread(&header, sizeof(RomHeader), 1, file) == 1) {
        printf("PRG ROM Size: %d KB\n", header.prg_rom_size * 16);
    }
}
```

BSCF <-- Buffer, size, count, file-pointer

---

```c
#include <stdio.h>
#include <stdint.h>

typedef struct {
    char magic[4];
    uint8_t prg_rom_size;
    uint8_t chr_rom_size;
    uint16_t flags;
} RomHeader;

void load_header(FILE* file) {
    RomHeader header;
    size_t items_read = fread(&header, sizeof(RomHeader), 1, file)
    if (items_read == 1) {
        printf("PRG Rom Size: %d KB\n", header.prg_rom_size);
    }
}
```

---